In [0]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.streaming.StreamingQuery
import play.api.libs.json._
import play.api.libs.functional.syntax._
import com.databricks.dais2025.SensorDataGenerator
import org.apache.spark.sql.execution.streaming.functions.current_batch_id // EDGE

object HelperFunctions {

  // ---------------------------
  // Latency parsing
  // ---------------------------
  case class LatencyMetrics(P0: Long, P50: Long, P90: Long, P95: Long, P99: Long)
  case class Latencies(
    processingLatencyMs: Option[LatencyMetrics],
    sourceQueuingLatencyMs: Option[LatencyMetrics],
    e2eLatencyMs: Option[LatencyMetrics]
  )
  case class StateOperatorMetrics(
    operatorName: String,
    numRowsTotal: Long,
    numRowsUpdated: Long,
    numRowsRemoved: Long,
    commitTimeMs: Long,
    memoryUsedBytes: Long,
    customMetrics: Map[String, Long]
  )

  // JSON readers
  implicit val latencyMetricsReads: Reads[LatencyMetrics] = Json.reads[LatencyMetrics]
  implicit val latenciesReads: Reads[Latencies] = Json.reads[Latencies]
  implicit val stateOperatorMetricsReads: Reads[StateOperatorMetrics] = (
    (JsPath \ "operatorName").read[String] and
    (JsPath \ "numRowsTotal").readWithDefault[Long](0L) and
    (JsPath \ "numRowsUpdated").readWithDefault[Long](0L) and
    (JsPath \ "numRowsRemoved").readWithDefault[Long](0L) and
    (JsPath \ "commitTimeMs").readWithDefault[Long](0L) and
    (JsPath \ "memoryUsedBytes").readWithDefault[Long](0L) and
    (JsPath \ "customMetrics").readWithDefault[Map[String, Long]](Map.empty)
  )(StateOperatorMetrics.apply _)


  // ---------------------------
  // Core Functions
  // ---------------------------

  /** Parses the JSON field from StreamingQueryProgress into Latencies + State Operators */
  def parseProgressMetrics(progress: org.apache.spark.sql.streaming.StreamingQueryProgress): (Option[Latencies], Seq[StateOperatorMetrics]) = {
    val jsValue = Json.parse(progress.json)

    val latenciesOpt = (jsValue \ "latencies").asOpt[Latencies]
    val stateOps = (jsValue \ "stateOperators").asOpt[Seq[StateOperatorMetrics]].getOrElse(Seq.empty)

    (latenciesOpt, stateOps)
  }


  /** Pretty prints latency metrics if present */
  def printLatencies(latenciesOpt: Option[Latencies]): Unit = {
    def fmt(label: String, opt: Option[LatencyMetrics]): String = opt match {
      case Some(m) => f"$label%-28s P50=${m.P50}%-8d P90=${m.P90}%-8d P95=${m.P95}%-8d P99=${m.P99}%-8d"
      case None    => f"$label%-28s N/A"
    }

    latenciesOpt match {
      case Some(lat) =>
        println("Latencies:")
        println("  " + fmt("E2E Latency (ms):", lat.e2eLatencyMs))
        println("  " + fmt("Processing Latency (ms):", lat.processingLatencyMs))
        println("  " + fmt("Source Queuing Latency (ms):", lat.sourceQueuingLatencyMs))
      case None =>
        println("Latencies: N/A")
    }
  }

  /** Main entry point — prints everything */
  def printRTMStreamMetrics(
    progressSeq: Seq[org.apache.spark.sql.streaming.StreamingQueryProgress],
    printLatenciesFlag: Boolean = true,
    printTransformWithStateMetricsFlag: Boolean = false
  ): Unit = {
    progressSeq.foreach { progress =>
      val batchId = progress.batchId
      val processedRows = progress.processedRowsPerSecond
      println(s"\nBatchId: $batchId | ProcessedRows/s: $processedRows")

      val (latenciesOpt, stateOps) = parseProgressMetrics(progress)

      if (printLatenciesFlag) {
        printLatencies(latenciesOpt)
      }
      
      if (printTransformWithStateMetricsFlag) {
        println("Not Added Yet")
        // printTransformWithStateMetrics(stateOps)
      }
    }
  }

  def getLastProgress(query: org.apache.spark.sql.streaming.StreamingQuery): Option[org.apache.spark.sql.streaming.StreamingQueryProgress] = {
    Option(query.lastProgress)
  }

  def stopStreamAfterBatchesCollectProgress(
    spark: SparkSession,
    stream: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row],
    queryName: String,
    maxBatches: Int,
    trigger: org.apache.spark.sql.streaming.Trigger,
    checkpointLocation: String,
    tableName: String
  ): Seq[org.apache.spark.sql.streaming.StreamingQueryProgress] = {
    import org.apache.spark.sql.functions._

    val triggerType = trigger match {
      case t if t.toString.toLowerCase.contains("realtime") => "realtime"
      case t if t.toString.toLowerCase.contains("processingtime") => "processingtime0"
      case _ => "unknown"
    }

    // Add sink_timestamp, triggertype, and batchId columns for latency calculation
    val dfWithMetadata = stream
      .withColumn("sink_timestamp", current_timestamp())
      .withColumn("triggertype", lit(triggerType))
      .withColumn("batch_id", current_batch_id())

    val writeStream = dfWithMetadata.writeStream
      .format("memory")
      .queryName(tableName)
      .trigger(trigger)
      .option("checkpointLocation", checkpointLocation)
      .outputMode("update")

    // RTM requires at-least-once delivery mode for memory sink
    val configuredStream = trigger match {
      case t if t.toString.toLowerCase.contains("realtime") =>
        writeStream.option("mode", "atLeastOnce")
      case _ => writeStream
    }

    val query = configuredStream.start()

    var lastBatchId = -1L
    var batchCount = 0
    val progressBuffer = scala.collection.mutable.Buffer[org.apache.spark.sql.streaming.StreamingQueryProgress]()
    while (query.isActive && batchCount < maxBatches) {
      getLastProgress(query).foreach { s =>
        if (s.batchId > lastBatchId) {
          lastBatchId = s.batchId
          batchCount += 1
          progressBuffer += s
          println("\nPretty JSON for last StreamingQueryProgress:")
          println(Json.prettyPrint(Json.parse(s.json)))
        }
      }
      Thread.sleep(200)
    }
    if (query.isActive) query.stop()
    progressBuffer.toSeq
  }

  /** Calculate latency metrics from a memory sink table using timestamp subtraction */
  def calculateLatencyFromMemoryTable(
    spark: SparkSession,
    tableName: String
  ): org.apache.spark.sql.DataFrame = {
    import org.apache.spark.sql.functions._

    spark.sql(s"SELECT * FROM $tableName")
      .withColumn("source_timestamp_ms", unix_millis(col("timestamp")))
      .withColumn("sink_timestamp_ms", unix_millis(col("sink_timestamp")))
      .withColumn("latency", col("sink_timestamp_ms") - col("source_timestamp_ms"))
  }
}